In [0]:
%sql
select *
from prod_latam_catalog.crm_reporting.dim_gdm_brand_profile
where brand_mdm_id = '3ebb2915d158a519c6e5991a98d78d75' 
limit 10

In [0]:
%sql
select distinct fragrance_priority
from prod_latam_catalog.crm_reporting.dim_gdm_brand_profile


In [0]:
%sql
select distinct skin_type
from prod_latam_catalog.crm_reporting.dim_gdm_brand_profile


In [0]:
'[{"analysis_clinical_sign":[{"clinical_sign":null,"normalized_score":null,"provider":null,"raw_score":null,"raw_score_range":null,"sign_range":null,"sign_type":null,"zone":null}],"analysis_concern":[{"active":null,"calculated":null,"concern":null,"concern_range":null,"concern_type":null,"provider":null,"value":null,"zone":null}],"analysis_type":null,"calculated_age":null,"declared_age":null,"exact_age":null,"location":null,"touchpoint":null}]'

In [0]:
%sql
select *
from prod_latam_catalog.crm_reporting.dim_gdm_brand_profile_hist
where brand_mdm_id = '3ebb2915d158a519c6e5991a98d78d75'
limit 10

In [0]:
from pyspark.sql.functions import col,count

gdm = spark.table("crm_reporting.dim_gdm_brand_profile")
# print(gdm.count())
# print(len(gdm.columns))

In [0]:
from pyspark.sql.functions import col,count

# Filtrar columnas sin NINGUN nulo
non_null_columns = [c for c in gdm.columns if gdm.filter(col(c).isNull()).count() == 0]

# Mostrar las columnas sin nulos
gdm_non_null = gdm[non_null_columns]
print(len(gdm_non_null.columns))


In [0]:
# Número total de filas en el DataFrame
total_rows = gdm.count()

# Crear una lista con columnas que no son completamente nulas
non_completely_null_columns = [c for c in gdm.columns if gdm.select(count(col(c).isNotNull())).collect()[0][0] > 0]


In [0]:
from pyspark.sql.functions import col, count, when

# Contar valores no nulos para cada columna
agg_expr = [count(when(col(c).isNotNull(), c)).alias(c) for c in gdm.columns]

# Ejecutar la agregación
non_null_counts = gdm.agg(*agg_expr).collect()[0]

# Filtrar columnas que no son completamente nulas
non_completely_null_columns = [c for c in gdm.columns if non_null_counts[c] > 0]

# Mostrar las columnas que no están completamente vacías
print(len(non_completely_null_columns))
print(non_completely_null_columns)

77
['brand_code', 'brand_country', 'mdm_source', 'brand_mdm_id', 'left_eye_color', 'right_eye_color', 'skin_tone', 'skin_undertone', 'lips_description', 'beauty_profile_modified_dt', 'analysis', 'analysis_modified_dt', 'external_factor', 'goal', 'misc_last_modified_dt', 'fragrance_priority', 'fragrance_preference_modified_dt', 'fragrance_routine', 'fragrance_routine_modified_dt', 'desired_color_finish', 'desired_color_permanence', 'desired_cover', 'hair_color_pref_modified_dt', 'hair_length', 'hair_state', 'hair_texture', 'hair_type', 'scalp_type', 'dandruff_frequency', 'hair_loss_volume', 'natural_hair_color', 'last_hair_style_look', 'hair_profile_modified_dt', 'hair_routine_heating', 'heating_last_modified_dt', 'hair_wash_frequency', 'hair_condition_frequency', 'last_hair_color_service', 'hair_color_service_frequency', 'scalp_sensitivity', 'hair_routine_modified_dt', 'desired_style', 'desired_hold', 'hair_style_pref_modified_dt', 'hair_care_product_used', 'products_used_modified_dt',

In [0]:
print(len(non_completely_null_columns))
#display(non_completely_null_columns.limit(10))

77


In [0]:
handling = spark.sql(
    """
    """)

In [0]:
%sql
-- PRIMER CRUCE
select --distinct
  -- count(1), min(to_date(created_dt)), max(to_date(created_dt))
  a.acq_source,
  --a.acq_sub_source,
  a.registration_source,
  a.registration_sub_source,
  a.created_dt,
  a.source_customer_id,
  b.brand_customer_id--,
  -- c.*
from prod_latam_catalog.crm_reporting.dim_customer a 
inner join prod_latam_catalog.crm_reporting.dim_customer_bridge b
  on a.source_customer_id = b.source_customer_id 
  and a.brand_code = b.brand_code 
  and a.brand_country = b.brand_country
-- inner join prod_latam_catalog.crm_reporting.dim_gdm_brand_profile c
--   on b.brand_customer_id = c.brand_mdm_id 
--   and b.brand_code = c.brand_code 
--   and b.brand_country = c.brand_country
where upper(a.source_name) IN ('DEMANDWARE') 
  AND lower(a.registration_sub_source) IN ('skin dr','hair quiz','product finder')
  AND a.brand_code IN ('DMC')
  AND a.brand_country IN ('BRA')
  AND to_date(a.created_dt) >= '2023-01-01'
  AND a.source_customer_id = 'ea12525eca20b6dbc024521312a350a7'
  --AND to_date(created_dt) BETWEEN '2023-01-01' AND '2023-12-31'
--group by all

In [0]:
%sql
-- PRIMER CRUCE
select --distinct
  -- count(1), min(to_date(created_dt)), max(to_date(created_dt))
  a.acq_source,
  --a.acq_sub_source,
  a.registration_source,
  a.registration_sub_source,
  a.created_dt,
  a.source_customer_id,
  b.brand_customer_id,
  c.*
from prod_latam_catalog.crm_reporting.dim_customer a 
inner join prod_latam_catalog.crm_reporting.dim_customer_bridge b
  on a.source_customer_id = b.source_customer_id 
  and a.brand_code = b.brand_code 
  and a.brand_country = b.brand_country
inner join prod_latam_catalog.crm_reporting.dim_gdm_brand_profile c
  on b.brand_customer_id = c.brand_mdm_id 
  and b.brand_code = c.brand_code 
  and b.brand_country = c.brand_country
where upper(a.source_name) IN ('DEMANDWARE') 
  -- AND lower(a.registration_sub_source) IN ('skin dr','hair quiz','product finder')
  AND lower(a.registration_sub_source) IN ('hair quiz')
  AND a.brand_code IN ('KER')
  AND a.brand_country IN ('COL')
  AND to_date(a.created_dt) >= '2023-01-01'
  --AND a.source_customer_id = 'ea12525eca20b6dbc024521312a350a7'
  --AND to_date(created_dt) BETWEEN '2023-01-01' AND '2023-12-31'
--group by all

In [0]:
%sql
select *
from prod_latam_catalog.crm_reporting.dim_gdm_brand_profile
where brand_code IN ('KER')
  AND brand_country IN ('COL')

In [0]:
%sql
select distinct category
 --desired_style_finish
from prod_latam_catalog.crm_reporting.dim_gdm_brand_profile
